# AI API Colab Host

GPU deployment notebook for the monorepo AI orchestration API. Run on a T4 GPU runtime. Secrets are requested interactively and are never stored in the notebook.

In [ ]:
# Clone the monorepo and install only AI API dependencies.
%cd /content
!git clone https://github.com/sea-hackathon-2026/ai-service.git
%cd /content/ai-service
!apt-get update -y >/dev/null
!apt-get install -y ffmpeg >/dev/null
!pip install -q -r requirements/ai-api.txt pyngrok nest_asyncio

In [ ]:
# Optional Wav2Lip setup. Skip this cell when using managed video generation.
from pathlib import Path

wav2lip_dir = Path('/content/Wav2Lip')
if not wav2lip_dir.exists():
    !git clone -q https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
    !mkdir -p /content/Wav2Lip/face_detection/detection/sfd
    !wget -q https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth
print('Wav2Lip source ready:', wav2lip_dir)

In [ ]:
# Configure runtime secrets and model adapters.
import getpass
import os

os.environ['AI_API_DEBUG'] = 'true'
os.environ['AI_API_STORAGE_LOCAL_PATH'] = '/content/ai-service/data/outputs'
os.environ['AI_API_API_KEY_SECRET'] = getpass.getpass('Client API key: ')
os.environ['AI_API_GEMINI_API_KEY'] = getpass.getpass('Gemini API key (optional): ')
os.environ['AI_API_LIVESTREAM_ENABLE_WAV2LIP'] = 'false'
os.environ['AI_API_WAV2LIP_DIR'] = '/content/Wav2Lip'
os.environ['AI_API_WAV2LIP_CHECKPOINT'] = '/content/Wav2Lip/checkpoints/Wav2Lip-SD-GAN.pt'

In [ ]:
# Start the API and publish port 8000 through ngrok.
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()
ngrok_token = getpass.getpass('ngrok token (optional): ')
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
public_url = ngrok.connect(8000).public_url
print('API:', public_url)
print('OpenAPI:', f'{public_url}/docs')
uvicorn.run('services.ai_api.main:app', host='0.0.0.0', port=8000)